In [6]:
import pandas as pd
import random
import math
from collections import Counter

#Charger les données
data = pd.read_csv("../data/diabetes.csv")
# On définit la colonne target (ce qu'on veut prédire)
# ici c'est Outcome (0 ou 1)
target_col = data.columns[-1]
#Les features = toutes les colonnes sauf la dernière
features = list(data.columns[:-1])

#ici on mesure le désordre des données de notre dataset
def entropy(data):
    # on récupère toutes les valeurs de la target (0 ou 1)
    labels = data[target_col].tolist()
    # nombre total de lignes
    total = len(labels)
    # compter combien de 0 et de 1(ça forme un dictionnaire avec la clé est 0 ou 1 et leurs valeurs sont combien de fois ils apparaient dans outcome)
    counts = Counter(labels)

    ent = 0
    # pour chaque classe (0 et 1)
    for c in counts:
        # calcul de la probabilité(ici on accède à la valeur de la classe)
        p = counts[c] / total
        # formule de l'entropie
        ent -= p * math.log2(p)

    return ent
    
#Cette fonction permet de diviser le dataset en deux sous-ensembles selon une condition sur une feature
#afin de construire les branches de l’arbre de décision.
def split_data(data, feature, split_value):
    #Ici on prend toutes les lignes où la valeur de la colonne (feature)
    # est inférieure ou égale au split value
    # ça forme le groupe gauche (left)
    left = data[data[feature] <= split_value]
    # Ici on prend toutes les lignes où la valeur de la colonne
    # est strictement supérieure au split value
    # ça forme le groupe droite (right)
    right = data[data[feature] > split_value]
    # On retourne les deux groupes
    return left, right

def best_split(data):
    #On calcule l'entropie du dataset avant de faire une division
    #ça représente le désordre initial
    base_entropy = entropy(data)

    # Initialisation des meilleures valeurs
    best_gain = -1
    best_feature = None
    best_split_value  = None
    # On teste chaque feature (chaque colonne)
    for feature in features:
        # on prend toutes les valeurs différentes de cette colonne
        # pour tester avec quoi on peut couper les données
        values = sorted(data[feature].unique())
        # On teste chaque valeur comme seuil
        for v in values:
            # On divise les données en deux groupes
            left, right = split_data(data, feature, v)

            # Si un des groupes est vide, on ignore ce cas
            # ça ne sert à rien de couper si tout est d’un seul côté
            if len(left) == 0 or len(right) == 0:
                continue
            #Calcul des proportions des groupes
            p_left = len(left) / len(data)
            p_right = len(right) / len(data)
            # Calcul de l'entropie après division
            # moyenne pondérée des entropies des deux groupes
            new_entropy = p_left * entropy(left) + p_right * entropy(right)
            # Calcul du gain d'information
            gain = base_entropy - new_entropy
            # Si ce gain est meilleur que les précédents
            if gain > best_gain:
                #on met à jour les meilleures valeurs
                best_gain = gain
                best_feature = feature
                best_split_value = v
    # À la fin, on retourne la meilleure feature et le meilleur seuil
    return best_feature, best_split_value

def build_tree(data, depth=0, max_depth=3):

    # on récupère toutes les valeurs de la target (0 ou 1)
    labels = data[target_col].tolist()

    # condition d'arrêt
    # si toutes les valeurs sont les mêmes (pur)
    # OU si on a atteint la profondeur max
    if len(set(labels)) == 1 or depth == max_depth:
        # on retourne la classe la plus fréquente (0 ou 1)
        return Counter(labels).most_common(1)[0][0]

    # on cherche la meilleure colonne et la meilleure valeur pour couper
    feature, split_value = best_split(data)

    # on divise les données en deux groupes
    left, right = split_data(data, feature, split_value)

    # on construit l’arbre récursivement
    # gauche et droite deviennent des sous-arbres
    return {
        "feature": feature,        # colonne utilisée pour la décision
        "split_value": split_value,    # valeur de séparation
        # on continue à construire l’arbre avec les données du côté gauche
        # on descend d’un niveau (depth+1)
        "left": build_tree(left, depth+1, max_depth),   # sous-arbre gauche
        # on continue à construire l’arbre avec les données du côté droit
        # on descend d’un niveau (depth+1)
        "right": build_tree(right, depth+1, max_depth)  # sous-arbre droite
    }
def random_dataset(data):
     #nombre de lignes dans le dataset
    n = len(data)
    # liste vide pour stocker le nouveau dataset
    sample = []
    # on va créer un dataset de même taille (n lignes)
    for _ in range(n):
        # choisir un index au hasard avec remise:(on peut reprendre plusieurs fois la même ligne)
        i = random.randint(0, n-1)
        
        # ajouter la ligne correspondante
        # data.iloc[i] = prendre la ligne i
        sample.append(data.iloc[i])

    # transformer la liste en DataFrame pandas
    return pd.DataFrame(sample)

# cette fonction crée plusieurs arbres pour former une forêt (Random Forest)
def train_forest(data, n_trees=5, max_depth=3):
    # liste vide pour stocker tous les arbres
    forest = []
    # on va créer plusieurs arbres (selon n_trees)
    for _ in range(n_trees):
        # créer un dataset aléatoire (avec remise)
        sample = random_dataset(data)
        
        # construire un arbre de décision avec ce dataset
        tree = build_tree(sample, max_depth=max_depth)
        
        # ajouter cet arbre dans la forêt
        forest.append(tree)

    # retourner la liste de tous les arbres
    return forest

def predict_tree(tree, row):

    # si on arrive à une feuille (valeur finale 0 ou 1)
    # ça veut dire qu'on a terminé la décision
    if tree in [0, 1]:
        return tree

    # récupérer la colonne utilisée pour la décision
    feature = tree["feature"]
    
    # récupérer la valeur de séparation
    split_value = tree["split_value"]

    # vérifier la condition
    # si la valeur de la ligne est <= seuil
    if row[feature] <= split_value:
        # aller à gauche de l’arbre
        return predict_tree(tree["left"], row)
    else:
        # aller à droite de l’arbre
        return predict_tree(tree["right"], row)

def predict_forest(forest, row):
    #liste pour stocker les prédictions de chaque arbre
    preds = []
    # pour chaque arbre dans la forêt
    for tree in forest:
        # faire une prédiction avec cet arbre
        #predict_tree suit l’arbre et retourne 0 ou 1
        preds.append(predict_tree(tree, row))

    # prendre la prédiction la plus fréquente (vote majoritaire)
    return Counter(preds).most_common(1)[0][0]
# on passe au test:
# créer la forêt (plusieurs arbres)
# ici on construit un modèle Random Forest avec 5 arbres
forest = train_forest(data, n_trees=5, max_depth=3)

# message pour dire que le modèle est prêt
print("Forest built!")

# test rapide sur les 5 premières lignes du dataset
for i in range(5):
    # prendre une ligne du dataset
    row = data.iloc[i]
    # faire une prédiction avec la forêt
    pred = predict_forest(forest, row)
    # afficher la prédiction et la vraie valeur
    print("Prediction:", pred, "Real:", row[target_col])

# initialiser compteur des bonnes prédictions
correct = 0
# tester le modèle sur tout le dataset
for i in range(len(data)):
    # prendre une ligne
    row = data.iloc[i]
    # faire la prédiction
    pred = predict_forest(forest, row)
    # comparer avec la vraie valeur
    if pred == row[target_col]:
        correct += 1
# calcul de la précision (accuracy)
accuracy = correct / len(data)
# afficher le résultat final
print("Accuracy:", accuracy)

Forest built!
Prediction: 1.0 Real: 1.0
Prediction: 0.0 Real: 0.0
Prediction: 1.0 Real: 1.0
Prediction: 0.0 Real: 0.0
Prediction: 0.0 Real: 1.0
Accuracy: 0.7552083333333334
